# Laboratory Assignment 8: High-Performance Big Data Analytics Using R

**Name:** Siddhanth Mungekar  
**Roll No:** 23102B0030  
**Branch:** CMPN/B

This notebook implements the NYC TLC Yellow Taxi analysis using data.table, functional programming, vectorization, foreach/doParallel, benchmarking, and ggplot2.

## 1. Install and Load Packages

In [ ]:
packages <- c("data.table","ggplot2","foreach","doParallel","microbenchmark","purrr","lubridate")
installed <- rownames(installed.packages())
for (p in packages) if (!p %in% installed) install.packages(p, repos="https://cloud.r-project.org")

library(data.table)
library(ggplot2)
library(foreach)
library(doParallel)
library(microbenchmark)
library(purrr)
library(lubridate)

## 2. Data Acquisition

Upload the NYC TLC Yellow Taxi CSV/CSV.GZ file and Taxi Zone Lookup CSV into the Colab/R working directory.

In [ ]:
list.files(pattern="\\.(csv|gz)$", ignore.case=TRUE)

# Update these filenames to match your uploaded files.
trip_file <- "yellow_tripdata.csv"
zone_file <- "taxi_zone_lookup.csv"

if (!file.exists(trip_file)) stop(paste("Trip file not found:", trip_file))
if (!file.exists(zone_file)) stop(paste("Zone file not found:", zone_file))

trips <- fread(trip_file)
zones <- fread(zone_file)
cat("Trip rows:", nrow(trips), "\n")
cat("Trip columns:", ncol(trips), "\n")

## 3. Inspect and Prepare Data

In [ ]:
str(trips)
dim(trips)
summary(trips)
head(trips)

In [ ]:
trips[, tpep_pickup_datetime := ymd_hms(tpep_pickup_datetime, quiet=TRUE, tz="America/New_York")]
trips[, tpep_dropoff_datetime := ymd_hms(tpep_dropoff_datetime, quiet=TRUE, tz="America/New_York")]

before <- nrow(trips)
trips <- unique(trips)
cat("Duplicates removed:", before - nrow(trips), "\n")

trips <- trips[
  !is.na(tpep_pickup_datetime) & !is.na(tpep_dropoff_datetime) &
  !is.na(trip_distance) & trip_distance >= 0 &
  !is.na(fare_amount) & fare_amount >= 0 &
  !is.na(total_amount) & total_amount >= 0 &
  !is.na(PULocationID) & !is.na(DOLocationID)
]

trips[, hour := hour(tpep_pickup_datetime)]
trips[, day := day(tpep_pickup_datetime)]
trips[, day_of_week := wday(tpep_pickup_datetime, label=TRUE, abbr=FALSE)]
trips[, month := month(tpep_pickup_datetime, label=TRUE, abbr=FALSE)]
trips[, pickup_date := as.Date(tpep_pickup_datetime)]

cat("Clean rows:", nrow(trips), "\n")

## 4. Join Taxi Zone Lookup

In [ ]:
setnames(zones, names(zones), make.names(names(zones)))
zone_id <- names(zones)[tolower(names(zones)) %in% c("locationid","location_id")][1]
zone_name <- names(zones)[tolower(names(zones)) == "zone"][1]
borough <- names(zones)[tolower(names(zones)) == "borough"][1]

zones_small <- zones[, .(LocationID=get(zone_id), Zone=get(zone_name), Borough=get(borough))]
pu <- copy(zones_small); setnames(pu, c("LocationID","Zone","Borough"), c("PULocationID","PUZone","PUBorough"))
do <- copy(zones_small); setnames(do, c("LocationID","Zone","Borough"), c("DOLocationID","DOZone","DOBorough"))

trips <- pu[trips, on="PULocationID"]
trips <- do[trips, on="DOLocationID"]
head(trips[, .(PULocationID,PUZone,PUBorough,DOLocationID,DOZone,DOBorough)])

## 5. Transportation Analytics

In [ ]:
trips_by_hour <- trips[, .(trips=.N), by=hour][order(hour)]
trips_by_day <- trips[, .(trips=.N), by=day_of_week][order(-trips)]
trips_by_month <- trips[, .(trips=.N), by=month]
fare_revenue <- trips[, .(avg_fare=mean(fare_amount,na.rm=TRUE), total_revenue=sum(total_amount,na.rm=TRUE), trips=.N), by=month]
top_routes <- trips[, .(trips=.N), by=.(PUZone,DOZone)][order(-trips)][1:10]
top_revenue_routes <- trips[, .(total_revenue=sum(total_amount,na.rm=TRUE)), by=.(PUZone,DOZone)][order(-total_revenue)][1:10]
payment_summary <- trips[, .(trips=.N,revenue=sum(total_amount,na.rm=TRUE)), by=payment_type][order(-trips)]

print(trips_by_hour)
print(trips_by_day)
print(trips_by_month)
print(fare_revenue)
print(top_routes)
print(top_revenue_routes)
print(payment_summary)

In [ ]:
distance_fare <- trips[, .(correlation=cor(trip_distance,fare_amount,use="complete.obs"), avg_distance=mean(trip_distance,na.rm=TRUE), avg_fare=mean(fare_amount,na.rm=TRUE))]
print(distance_fare)

## 6. Functional, Vectorized and data.table Implementations

In [ ]:
hourly_lapply <- function(x) as.integer(unlist(lapply(0:23, function(h) sum(x==h,na.rm=TRUE))))
hourly_vectorized <- function(x) as.integer(table(factor(x,levels=0:23)))
hourly_datatable <- function(dt) dt[, .N, by=hour][order(hour)]$N
hourly_map <- function(x) map_int(0:23, ~sum(x==.x,na.rm=TRUE))

comparison <- data.table(hour=0:23,
  lapply=hourly_lapply(trips$hour),
  vectorized=hourly_vectorized(trips$hour),
  data_table=hourly_datatable(trips),
  purrr_map=hourly_map(trips$hour))
print(comparison)
cat("All implementations agree:", all(comparison$lapply==comparison$vectorized & comparison$vectorized==comparison$data_table & comparison$data_table==comparison$purrr_map), "\n")

## 7. Performance Benchmarking

In [ ]:
bench <- microbenchmark(
  lapply=hourly_lapply(trips$hour),
  vectorized=hourly_vectorized(trips$hour),
  data_table=hourly_datatable(trips),
  purrr_map=hourly_map(trips$hour),
  times=5L)
print(bench)
benchmark_summary <- summary(bench)[,c("expr","min","median","mean","max")]
print(benchmark_summary)

## 8. Sequential vs Parallel Processing

In [ ]:
months <- unique(trips$month)
months <- months[!is.na(months)]

seq_time <- system.time({
  sequential <- lapply(months, function(m) sum(trips[month==m,total_amount],na.rm=TRUE))
})

cores <- max(1, parallel::detectCores()-1)
cl <- makeCluster(cores)
registerDoParallel(cl)

parallel_time <- system.time({
  parallel_result <- foreach(m=months,.combine=c,.packages="data.table") %dopar% {
    sum(trips[month==m,total_amount],na.rm=TRUE)
  }
})

stopCluster(cl)
registerDoSEQ()

speedup <- as.numeric(seq_time["elapsed"])/as.numeric(parallel_time["elapsed"])
comparison_parallel <- data.table(
  approach=c("Sequential","Parallel"),
  elapsed_seconds=c(as.numeric(seq_time["elapsed"]),as.numeric(parallel_time["elapsed"]))
)
print(comparison_parallel)
cat("Cores:",cores,"\nSpeedup:",round(speedup,3),"x\n")

## 9. Data Visualizations

In [ ]:
ggplot(trips_by_hour,aes(hour,trips))+geom_col()+labs(title="Taxi Trips by Hour",x="Hour of Day",y="Number of Trips")+theme_minimal()

In [ ]:
ggplot(trips_by_day,aes(reorder(day_of_week,trips),trips))+geom_col()+coord_flip()+labs(title="Taxi Demand by Day of Week",x="Day",y="Number of Trips")+theme_minimal()

In [ ]:
ggplot(fare_revenue,aes(month,avg_fare,group=1))+geom_line()+geom_point()+labs(title="Average Fare by Month",x="Month",y="Average Fare")+theme_minimal()

In [ ]:
set.seed(42)
s <- if(nrow(trips)>100000) trips[sample(.N,100000)] else trips
ggplot(s,aes(trip_distance,fare_amount))+geom_point(alpha=.25)+labs(title="Trip Distance vs Fare Amount",x="Trip Distance",y="Fare Amount")+theme_minimal()

In [ ]:
payment_plot <- trips[, .N, by=payment_type][order(-N)]
ggplot(payment_plot,aes(factor(payment_type),N))+geom_col()+labs(title="Payment Type Distribution",x="Payment Type",y="Number of Trips")+theme_minimal()

## 10. Critical Analysis and Conclusion

`data.table` is recommended for large-scale filtering, joining, grouping and aggregation because it is designed for efficient in-memory tabular processing. Vectorized operations are highly effective for simple calculations, while `lapply()`/`purrr::map()` are useful for repeated or modular operations. Parallel processing can improve runtime for sufficiently large independent workloads, but cluster setup and data-transfer overhead mean it is not always faster. The final recommendation should be supported by the benchmark values printed above.

### Conclusion
The experiment demonstrates an end-to-end scalable R workflow for NYC taxi data: preprocessing, zone integration, transportation analytics, functional/vectorized/data.table comparison, sequential versus parallel execution, benchmarking, and visualization.

## Dataset Source

NYC Taxi & Limousine Commission (TLC) – Yellow Taxi Trip Records:
https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page